# Entregable final — Cashea FPD
## Ronda 1 (parte de validación): sensibilidad de umbrales y bandas

**Idea central:** el modelo estima el riesgo; el negocio decide cuánto riesgo asumir.

Todo lo de esta sección sale de las predicciones **fuera de muestra** de la validación (enero, fold 2 del CV temporal) del modelo Gradient Boosting: `datos/validacion_fold2_gb.csv`. El test no se usa para elegir nada.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path

# Corré el notebook desde la carpeta notebooks/ (o ajustá RAIZ)
RAIZ = Path("..")
val = pd.read_csv(RAIZ / "datos" / "validacion_fold2_gb.csv")
y_val = val["y_val"].to_numpy()
p_val = val["p_val"].to_numpy()
UMBRAL = 0.155   # fijado desde validación
print(len(y_val), "filas | tasa FPD =", round(y_val.mean(), 4))

57426 filas | tasa FPD = 0.0831


### 1. Tabla de sensibilidad (umbral de 0,05 a 0,40, paso 0,005)

Un caso con `prob >= umbral` se va a revisión (positivo predicho).
- `pct_rechazo` = (tp+fp)/N
- `pct_fpd_evitado` = recall = tp/(tp+fn)
- `tasa_fpd_aprobados` = fn/(fn+tn)
- `precision` = tp/(tp+fp)

In [2]:
def fila_umbral(y, p, u):
    pred = p >= u
    tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum())
    fn = int((~pred & (y == 1)).sum()); tn = int((~pred & (y == 0)).sum())
    n = tp + fp + fn + tn
    return {
        "umbral": u,
        "pct_rechazo": (tp + fp) / n,
        "pct_fpd_evitado": tp / (tp + fn) if (tp + fn) else np.nan,
        "tasa_fpd_aprobados": fn / (fn + tn) if (fn + tn) else np.nan,
        "precision": tp / (tp + fp) if (tp + fp) else np.nan,
        "recall": tp / (tp + fn) if (tp + fn) else np.nan,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }

umbrales = np.round(np.arange(0.05, 0.40 + 1e-9, 0.005), 3)
sens = pd.DataFrame([fila_umbral(y_val, p_val, u) for u in umbrales])

assert len(sens) == 71 and 0.155 in sens["umbral"].values
assert (sens[["tp", "fp", "fn", "tn"]].sum(axis=1) == len(y_val)).all()
assert sens["pct_rechazo"].is_monotonic_decreasing

sens.to_csv(RAIZ / "datos" / "sensibilidad_validacion.csv", index=False)
sens.loc[sens["umbral"] == 0.155]

,umbral,pct_rechazo,pct_fpd_evitado,tasa_fpd_aprobados,precision,recall,tp,fp,fn,tn
21,0.155,0.096141,0.224319,0.071284,0.193805,0.224319,1070,4451,3700,48205


### 2. `corte_medio` y las tres bandas

`corte_medio` es la probabilidad del percentil 70 en validación. Sale de **esta** versión del modelo: si se reentrena, hay que recalcularlo.

In [3]:
corte_medio = float(np.percentile(p_val, 70))
print(f"corte_medio = {corte_medio:.4f}")
assert corte_medio < UMBRAL

banda = np.where(p_val >= UMBRAL, "alto", np.where(p_val >= corte_medio, "medio", "bajo"))
tabla_bandas = (pd.DataFrame({"banda": banda, "y": y_val})
    .groupby("banda")
    .agg(n=("y", "size"), fpd=("y", "sum"), tasa_fpd=("y", "mean"))
    .assign(pct_casos=lambda d: d["n"] / d["n"].sum(),
            pct_del_total_fpd=lambda d: d["fpd"] / d["fpd"].sum())
    .loc[["bajo", "medio", "alto"]])
display(tabla_bandas)
print("FPD que concentra el 30% más riesgoso:", round(y_val[p_val >= corte_medio].sum() / y_val.sum(), 3))

corte_medio = 0.1001


,n,fpd,tasa_fpd,pct_casos,pct_del_total_fpd
banda,,,,,
bajo,40198,2286,0.056869,0.699997,0.479245
medio,11707,1414,0.120782,0.203862,0.296436
alto,5521,1070,0.193805,0.096141,0.224319


FPD que concentra el 30% más riesgoso: 0.521


### 3. Verificación única en TEST (febrero–marzo 2025)

El umbral **0,155** y el `corte_medio` **0,1001** quedaron fijados en validación. **El test no se usa para elegir nada**: solo se mide, una vez, cómo se comportan esos valores con datos que el modelo no vio.

Las predicciones salen del modelo guardado (`modelo/modelo_fpd_cashea.joblib`, Gradient Boosting) y están en `datos/test_predicciones_gb.csv`.

In [4]:
from sklearn.metrics import confusion_matrix, roc_auc_score

test = pd.read_csv(RAIZ / "datos" / "test_predicciones_gb.csv")
y_test = test["y_test"].to_numpy()
p_test = test["p_test"].to_numpy()
CORTE_MEDIO = 0.1001

print(len(y_test), "filas | tasa FPD =", round(y_test.mean(), 4))
print("AUC test:", round(roc_auc_score(y_test, p_test), 4), "(reportado 0.6758)")

pred = p_test >= UMBRAL
tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
n = tn + fp + fn + tp
display(pd.DataFrame([[tn, fp], [fn, tp]], index=["real 0 (paga)", "real 1 (FPD)"],
                     columns=["pred 0 (aprobar)", "pred 1 (revisión)"]))

rechazo, recall, precision = (tp + fp) / n, tp / (tp + fn), tp / (tp + fp)
print(f"rechazo   : {rechazo:.1%}  (reportado 8,0%)")
print(f"recall    : {recall:.1%}  (FPD evitado, reportado 19,4%)")
print(f"precision : {precision:.1%}  (reportado 18,2%)")
assert round(rechazo, 3) == 0.080 and round(recall, 3) == 0.194 and round(precision, 3) == 0.182

104088 filas | tasa FPD = 0.0753
AUC test: 0.6758 (reportado 0.6758)


,pred 0 (aprobar),pred 1 (revisión)
real 0 (paga),89410,6835
real 1 (FPD),6321,1522


rechazo   : 8.0%  (reportado 8,0%)
recall    : 19.4%  (FPD evitado, reportado 19,4%)
precision : 18.2%  (reportado 18,2%)


**La matriz depende del umbral; el AUC no.** Cada umbral reparte distinto a los casos entre aprobar y revisar, así que `tp`, `fp`, `fn` y `tn` cambian. El AUC (0,6758) mide qué tan bien el modelo *ordena* a los clientes, evaluado sobre todos los umbrales a la vez: mover el umbral no lo toca. El AUC es una propiedad del modelo; la matriz es una consecuencia de la política.

A modo ilustrativo (no se elige nada con esto):

In [5]:
filas = []
for u in [0.12, UMBRAL, 0.20]:
    pr = p_test >= u
    a, b_, c_, d_ = confusion_matrix(y_test, pr, labels=[0, 1]).ravel()
    filas.append({"umbral": u, "tn": a, "fp": b_, "fn": c_, "tp": d_,
                  "rechazo": (d_ + b_) / len(y_test), "recall": d_ / (d_ + c_),
                  "AUC": round(roc_auc_score(y_test, p_test), 4)})
pd.DataFrame(filas)

,umbral,tn,fp,fn,tp,rechazo,recall,AUC
0,0.120,80217,16028,4988,2855,0.181414,0.364019,0.6758
1,0.155,89410,6835,6321,1522,0.080288,0.194058,0.6758
2,0.200,94374,1871,7320,523,0.023000,0.066684,0.6758


### 4. Las tres bandas en TEST (control)

Solo se reporta qué porcentaje cae en cada banda y su tasa de FPD. El modelo final se entrenó con un mes más de datos que el del fold 2 (que generó los cortes), así que los porcentajes no tienen por qué coincidir con los de validación.

In [6]:
banda_t = np.where(p_test >= UMBRAL, "alto", np.where(p_test >= CORTE_MEDIO, "medio", "bajo"))
tabla_bandas_test = (pd.DataFrame({"banda": banda_t, "y": y_test})
    .groupby("banda").agg(n=("y", "size"), fpd=("y", "sum"), tasa_fpd=("y", "mean"))
    .assign(pct_casos=lambda d: d["n"] / d["n"].sum())
    .loc[["bajo", "medio", "alto"]])
display(tabla_bandas_test)

,n,fpd,tasa_fpd,pct_casos
banda,,,,
bajo,76864,4146,0.053939,0.738452
medio,18867,2175,0.115281,0.181260
alto,8357,1522,0.182123,0.080288


### 5. Solicitudes de ejemplo (`datos/ejemplos.json`)

Tres solicitudes **reales de test** (febrero–marzo 2025), en formato crudo, con los campos del contrato de la API. Sirven para probar `POST /predecir` y `POST /predecir_lote`.

- Las variables crudas se reconstruyeron de las del modelo: `gmv = expm1(log_gmv)`, `anticipo = ratio_anticipo × gmv`, `producto = SPLIT_n`. Se verificó que, al pasarlas por `preparar_solicitud` y el modelo, reproducen **exactamente** la probabilidad que tenían en test (diferencia máxima 0,0).
- Banda alto: primera compra de *Primera necesidad* con ticket alto (`gmv` ≥ 73,42, cuartil superior de necesidad en train). En ese grupo de test, 269 de 485 casos caen en banda alto, 80 en medio y 136 en bajo.

In [7]:
import json
ejemplos = json.load(open(RAIZ / "datos" / "ejemplos.json", encoding="utf-8"))
campos = ["solicitud_id", "fecha_origen", "gmv", "anticipo", "producto", "score",
          "tipo_orden", "ordenes_previas", "comercio_id", "vertical"]
assert len(ejemplos) == 3 and all(list(e) == campos for e in ejemplos)
pd.DataFrame(ejemplos)

,solicitud_id,fecha_origen,gmv,anticipo,producto,score,tipo_orden,ordenes_previas,comercio_id,vertical
0,15427270,2025-02-09,160.00,32.00,SPLIT_12,405,Follow up order,1,qyDZM00IeQR8siGlx0IpaA==,Tecnología
1,16515008,2025-02-27,27.42,16.45,SPLIT_1,456,Follow up order,0,7pHC94OxVCo/XxhxusQtxQ==,Supermercados
2,14940582,2025-02-01,77.09,46.25,SPLIT_1,468,First order,0,NqfeCmO2UMrSDU8E+DVeWA==,Farmacias
